In [1]:
import random
import math
import copy

GRID_SIZE = 25

components = {
    "ALU": (5, 5),
    "Cache": (7, 4),
    "Control Unit": (4, 4),
    "Register File": (6, 6),
    "Decoder": (5, 3),
    "Floating Unit": (5, 5)
}

component_order = list(components.keys())

def generate_random_chromosome():
    return [(random.randint(0, GRID_SIZE - components[name][0]),
             random.randint(0, GRID_SIZE - components[name][1]))
            for name in component_order]

def initial_population(n=6):
    return [generate_random_chromosome() for _ in range(n)]

def get_center(coord, comp_name):
    x, y = coord
    w, h = components[comp_name]
    return (x + w/2, y + h/2)

def euclidean(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

connections = [("Register File", "ALU"),("Control Unit", "ALU"),("ALU", "Cache"),("Register File", "Floating Unit"),("Cache", "Decoder"),("Decoder", "Floating Unit")]

def compute_wiring_distance(chromosome):
    pos = dict(zip(component_order, chromosome))
    return sum(euclidean(get_center(pos[a], a), get_center(pos[b], b)) for a, b in connections)

def compute_bounding_area(chromosome):
    xs, ys = [], []
    for i, (x, y) in enumerate(chromosome):
        w, h = components[component_order[i]]
        xs.extend([x, x + w])
        ys.extend([y, y + h])
    return (max(xs) - min(xs)) * (max(ys) - min(ys))

def count_overlaps(chromosome):
    overlaps = 0
    for i in range(len(chromosome)):
        for j in range(i + 1, len(chromosome)):
            x1, y1 = chromosome[i]
            x2, y2 = chromosome[j]
            w1, h1 = components[component_order[i]]
            w2, h2 = components[component_order[j]]

            if not (x1 + w1 <= x2 or x1 >= x2 + w2 or y1 + h1 <= y2 or y1 >= y2 + h2):
                overlaps += 1
    return overlaps

def fitness(chromosome, alpha=1000, beta=2, gamma=1):
    overlap = count_overlaps(chromosome)
    wiring = compute_wiring_distance(chromosome)
    area = compute_bounding_area(chromosome)
    return -(alpha * overlap + beta * wiring + gamma * area)


def select_parents(pop, k=2):
    return random.sample(pop, k)

def single_point_crossover(p1, p2):
    point = random.randint(1, len(p1)-1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

def two_point_crossover(p1, p2):
    pt1 = random.randint(0, len(p1) - 2)
    pt2 = random.randint(pt1 + 1, len(p1) - 1)
    c1 = p1[:pt1] + p2[pt1:pt2] + p1[pt2:]
    c2 = p2[:pt1] + p1[pt1:pt2] + p2[pt2:]
    return c1, c2

def mutate(chromosome, mutation_rate=0.1):
    if random.random() < mutation_rate:
        index = random.randint(0, len(chromosome) - 1)
        w, h = components[component_order[index]]
        chromosome[index] = (random.randint(0, GRID_SIZE - w),
                              random.randint(0, GRID_SIZE - h))
    return chromosome

def genetic_algorithm(iterations=15, pop_size=6, crossover_method='single'):
    population = initial_population(pop_size)
    best = None

    for gen in range(iterations):
        population = sorted(population, key=lambda c: fitness(c))
        if best is None or fitness(population[0]) > fitness(best):
            best = copy.deepcopy(population[0])

        next_gen = [copy.deepcopy(population[0])]

        while len(next_gen) < pop_size:
            p1, p2 = select_parents(population)
            if crossover_method == 'single':
                c1, c2 = single_point_crossover(p1, p2)
            else:
                c1, c2 = two_point_crossover(p1, p2)
            next_gen.extend([mutate(c1), mutate(c2)])

        population = next_gen[:pop_size]

    return best

#For Task 2
optimal_layout = genetic_algorithm(crossover_method='single')
print("Optimal layout:", optimal_layout)
print("Total wiring distance:", compute_wiring_distance(optimal_layout))
print("Bounding box area:", compute_bounding_area(optimal_layout))
print("Pair wise overlap count:", count_overlaps(optimal_layout))
print("Total Fitness value:", fitness(optimal_layout))


Optimal layout: [(18, 4), (3, 9), (21, 18), (5, 10), (9, 2), (18, 15)]
Total wiring distance: 81.46649961254373
Bounding box area: 440
Pair wise overlap count: 2
Total Fitness value: -2602.9329992250873
